# Alaska: correcting the ionosphere in a NISAR pair

Track 87, frame 57, 2026-07-14 / 2026-07-26.

**This notebook is a walkthrough, not a runnable demonstration.**  A pair needs roughly
55 GB of RSLC downloads, 300 GB of free scratch, a CUDA GPU and several hours.  The cells
below are the commands in the order they are run, with a note on what each produces; run
them in a terminal, in a working directory of their own.  End to end this pair took about
2 h 20 m and 245 GB on one A100.

Everything reads `pair.yaml` beside this notebook, which carries the track, frame, dates
and the per-pair fill and unwrapping settings.

## 1. Screening: which pair is worth the RSLC chain

The GSLC products are geocoded and cheap, so the question of whether a pair carries
ionospheric streaking is settled before any RSLC is downloaded.  `stack` accumulates the
interferometric and power sums for every date pair once; `streak-index` turns that into
R_FFT, the angular concentration of the coherence loss, and theta, its orientation in radar
coordinates.

In [ ]:
# one pass over the GSLC stack, ~25 GB per date
!ionoNISAR download --track 87 --frame 57 --level GSLC --dates 20260620 20260702 20260714 20260726 20260819 20260831
!ionoNISAR stack    --track 87 --frame 57 --dates 20260620 20260702 20260714 20260726 20260819 20260831

`stack` writes one accumulation file and **reuses it on every later run** -- re-running the
cell prints `reusing accumulation ...` and does no work.  That file is all the later stages
need, so once it exists the GSLC granules themselves can be deleted, which is most of the
disk this stage costs.

In [ ]:
# R_FFT and theta at a 480 m cell, both frequencies
!ionoNISAR streak-index --track 87 --frame 57 --dates 20260620 20260702 20260714 20260726 20260819 20260831 \
    --rebin 2 --out gslc_streak/streak_index.csv

`select-pair` ranks the table.  R_FFT says how concentrated the coherence loss is in one
orientation, not how much coherence was lost, so it is a screening statistic: pairs above
about 0.5 generally show pronounced streaking, and that value is an empirical guide rather
than a threshold.  The sign of `D_RA` is the direction the anisotropic gap fill should
stretch along.

In [ ]:
!ionoNISAR select-pair gslc_streak/streak_index.csv --freq A --top 5

---

## 2. Inputs for the chosen pair

`pair.yaml` names the pair from here on.  The DEM comes from Copernicus GLO-30, the map
grid is an empty raster that pins every later product to one lattice, and the glacier mask
comes from the Randolph Glacier Inventory 7.0 -- offsets over ice measure ice flow, not
registration, so they are excluded and filled.

In [ ]:
!ionoNISAR download --track 87 --frame 57 --level RSLC --dates 20260714 20260726
!ionoNISAR --config pair.yaml dem
!ionoNISAR --config pair.yaml grid
!ionoNISAR --config pair.yaml glacier-mask

# what is present, what is missing, and whether this machine can run the rest
!ionoNISAR --config pair.yaml check

Each of these three keeps an input that already exists and says so, so the cell is safe to
re-run; pass `--force` to rebuild one deliberately.

`check` reports what is present, whether both granules carry frequency A **and** B for this
polarisation -- the split-spectrum screen differences the two carriers, so a product without
B cannot be corrected this way -- and how much of the frame polygon the DEM reaches.

### A note on per-pair settings

`pair.yaml` carries the fill and outlier settings this pair needs.  They are not universal:
the stiffer fit (`cutoff: 30`, `robust: 0`) exists because this frame carries a narrow,
steep streak that the default fill bridges, and the hole settings hand deep glacier
interiors over to a held surface rather than letting the continuation run away.

The split-spectrum stage also has a consistency gate (gate 8) that compares the unwrapped
phi_A against the wrapped interferogram at the screen scale.  If it fails, the run stops
rather than publishing a screen built on a bad unwrap.  The remedies, in order of
preference, are a different unwrapper (`split.unwrap_method`), `--split-phia-from gradient`
to build the screen on the integrated field instead, and only then relaxing the gate with
`split.gate_bracket` -- which should be recorded, because it changes what the gate certifies.

## 3. Registration

One command covers geometric coregistration (rdr2geo into the reference, geo2rdr into the
secondary, resample), dense offset estimation with PyCuAmpcor, masking and gap filling of
the offset field, the azimuth rubbersheet, the decision on Doppler-dependent refocusing,
and the interferogram.

The ionosphere shifts targets along track, and a single shift per pixel only registers the
centre of the Doppler band.  Where the applied field changes quickly along track, each
sub-aperture needs a different shift, and `--refocus-km auto` sweeps the pierce-point
aperture and applies the refocus only where it buys coherence.

This is the expensive step: **about 1.5 hours on one A100 and ~180 GB of scratch**, most of
it in the two dense correlation passes and the resampling.  It has to keep that scratch --
the phase-screen stage reads the exported pair and the geometry rasters back.

In [ ]:
!ionoNISAR --config pair.yaml coregister

`pair.yaml` names CUDA device 0.  On a shared machine that card is often busy, and
`coregister` forwards any argument it does not recognise to the tool underneath, so a
different device is just:

```
!ionoNISAR --config pair.yaml coregister --gpus 4
```

`--gpus 0,1,2,3` shards the correlation lattice over several cards, one process each.

Worth reading in the log: the `[coreg] check` residuals, which say whether the geometric
registration converged; the fill's anisotropy, which should be well above 1 for a
range-elongated field; and the `[refocus]` verdict, which reports the aperture chosen and
the coherence it gained, or says that there was nothing to refocus.

## 4. The ionospheric phase screen

The hybrid screen takes its long along-track wavelengths from the split-spectrum estimate
and its short ones from the azimuth offsets.  The two are complementary: the azimuth route
integrates along track, so its noise accumulates as a random walk and it drifts at long
wavelengths, while the split-spectrum solve integrates nothing but amplifies band-phase
noise by 1/|det|.

Asking for `hybrid` builds both parents first; neither is rebuilt if it is already on disk.
**About 45 minutes**, most of it the split-spectrum stage, which coregisters frequency B of
its own accord before it can difference the two bands.

Two lines in the log are the ones to read.  The `[gate]` line compares the unwrapped phi_A
against the wrapped interferogram's own gradients at the screen scale, and the run **stops**
rather than publishing a screen built on a bad unwrap.  The `[unwrap]` lines report how many
connected components the unwrapper returned and how much had to be re-referenced onto the
largest one's datum -- a frame that fragments into many components is the case to look at
before trusting the screen.

In [ ]:
!ionoNISAR --config pair.yaml screen --routes hybrid

## 5. Or the whole thing at once

In [ ]:
!ionoNISAR --config pair.yaml run --routes hybrid

## 6. What comes out

Under `outputs_unified/hybrid/`:

| file | what it is |
|---|---|
| `ifg_phase_geo_t087_20260714_20260726_rbcs.tif` | the corrected interferometric phase, geocoded |
| `ifg_coh_geo_t087_20260714_20260726_rbcs.tif` | its coherence |
| `iono_screen_t087_20260714_20260726.npz` | the hybrid screen on the offset lattice |
| `iono_screen_t087_20260714_20260726.tif` | the same screen, geocoded |

`outputs_unified/common/` holds the rungs the screen was applied to, so the correction can be
undone or compared: `_prerb` (coregistered only), `_rb` (+ azimuth rubbersheet) and `_rbc`
(+ carrier compensation).  `outputs_unified/offsets/` and `outputs_unified/split/` hold the two parent
screens, which the hybrid needs and which are worth looking at on their own.